In [ ]:
Content
1. Storage
2. ACID Properties
    - CAP Theorem 

# 1. Storage

In [ ]:
1. Structured
    - Rows/ Columns, pre-defined schema (ex: excel sheet)

2. Unstructrued
    - No pre-defined format (logs, images, videos)

#Categories of Storage
1. Database Storage
    - Structured or unstructured data managed by a DBMS.
    - SQL, NoSQL DBs
    - A database engine (e.g., PostgreSQL, MongoDB) persists data in a format optimized for queries, indexing, transactions, and consistency. 
        The storage engine (e.g., InnoDB, WiredTiger) handles how and where data is stored on disk.
    - Types
        - SQL (Relational DBs): Structured schema (PostgreSQL, MySQL, Oracle).
    	- NoSQL: Flexible schema (MongoDB, Cassandra, DynamoDB).
    - Characteristics
        - ACID/BASE consistency depending on type.
    	- Indexing, transactions, replication, etc.
    	- Internally may use file, block, or object storage.

2. Object Storage
    - Data is stored as objects containing the data itself, metadata, and a globally unique identifier. There’s no hierarchy; it’s flat.
    - Ex: amazon s3, google cloud storage
    - Storing backups, images, videos, logs, database snapshots

3. File Storage
    - Data is stored in a hierarchical structure (directories and files). Shared file systems allow multiple clients to access the same files.
    - Ex: NFS (Network File System), Amazon Elastic File System
    - Ex: Config Files, Semi-structured data (CSV, Parquet, JSON)

4. Block Storage
    - Block Storage (e.g., EBS, iSCSI) gives raw block-level storage volumes that are mounted like physical drives. It’s used where low-latency and high-throughput I/O is critical.
    - Ex: Amazon EBS (Elastic Block Store)
        - Storage layer for DBMS data files 




How They’re Used Together

In real systems, file and block storage often complement each other:
	•	A PostgreSQL database (running on an EC2) uses EBS block storage for its data directory.
	•	Logs and file uploads from the same app are saved to a shared NFS (file storage).
	•	A backup process writes PostgreSQL dumps to S3 (object storage).



| Storage Type       | Hierarchy        | Protocol/Access       | Common Use Cases                        | Example Systems        |
|--------------------|------------------|------------------------|------------------------------------------|------------------------|
| Database Storage   | Schema-based     | SQL/NoSQL DB Engines   | Transactional data, analytics            | PostgreSQL, MongoDB    |
| Object Storage     | Flat (object ID) | HTTP/S3 APIs           | Backups, media, logs                     | Amazon S3, GCS, Azure  |
| File Storage       | Filesystem       | NFS, SMB, POSIX        | Logs, intermediate ETL data              | NFS, EFS, Hadoop HDFS  |
| Block Storage      | None (raw blocks)| iSCSI, NVMe, EBS       | Primary DB storage, high IOPS workloads  | Amazon EBS, SAN, LUN   |



In [ ]:
# Propeties
Storage Properties
    - 

# 2. ACID Properties

In [ ]:
ACID is a set of properties that guarantee reliable and consistent transactions in a database system. These are essential for maintaining data integrity, especially in systems with concurrent users or failures.

The ACID guarantees come from the DBMS layer (Database engine), not raw storage. However, underlying storage must support durability and consistency:

1. A: Atomicity
    - All parts of a transaction succeed or none do. No partial writes.
    - Like an ATM: money is either withdrawn or not.
2. C: Consistency
    - Data must always remain in a valid state, respecting rules and constraints.
    - Balance can’t go negative unless allowed.
3. I: Isolation
    - Transactions occur independently, even if executed concurrently.
    - Two people can’t update the same row at once.
4. D: Durability
    - Once committed, data stays safe—even after crash/power failure.
    - Once receipt is printed, the transaction is final.

Example:        
BEGIN;
UPDATE accounts SET balance = balance - 100 WHERE id = 1;
UPDATE accounts SET balance = balance + 100 WHERE id = 2;
COMMIT;
	•	Atomicity: Both updates must happen or neither.
	•	Consistency: The sum of balances remains the same.
	•	Isolation: No other transfer can read a half-updated state.
	•	Durability: Once committed, it’s written to disk (e.g., via WAL).




In [ ]:
Trade-Off        
1. Scalability (can handle growth)
2. Reliability (resistant to failure)
3. Performance (speed of read/write)


# CAP Theorem
In any distributed system, you can only fully guarantee 2 of the 3:
1. Consistency - Every read receives the most recent write or an error. All clients view the same data, even after write/ delete.
    -  Strong guarantee, but can increase latency.
2. Availability - Every request receives a (non-error) response, without guarantee that it contains the most recent write.
    - The system continues to operate even if some nodes are down.
    - Highly fault-tolerant, fast responses.
    - But, it May return stale data during partition.
                           
3. Partition Tolerance - The system continues to operate despite network failure b/w the two nodes. (communication failures between nodes).
    - Required in any distributed system — networks will fail.
	- CAP theorem assumes P is non-negotiable in real-world systems.


# CAP Trade-Off
1. C+A: Partition Tolerance is loose: Works only when network is fully reliable (rare)
2. C+P: Availability is loose: System may reject requests (errors)
3. A+P Consistency:  May return stale data or allow conflicts


2. CP (Consistency + Partition Tolerance)
    - System may reject requests ( with errors)
    - Prioritizes data correctness over availability
During a partition, the system may reject requests to avoid inconsistent reads
Not always available, but when it is — data is guaranteed to be correct
Example: HBase: Strongly consistent. If a node can't confirm a write across replicas, it won't serve it — even if
it means being unavailable briefly.
When to use: Financial systems, banking apps, anything where data integrity is critical
                           
*No system can have all 3 at the same time